# C1-M2 — Segment3D class-agnostic backend (staged pilot)

**Protocol + PREDECLARED stopping rule: `docs/c1_m2_protocol.md`.** Stage 1 = room_2 ONLY; run frl/office/room_1 only if room_2 passes the gate.

**Scope:** raw `mesh.ply` → Segmentator graph segments → Segment3D masks → frozen `SegmentationOutput` sidecar → one tar.gz on Drive.

**Isolation (G2):** only `mesh.ply` + the three oracle-free segmenter files enter this runtime — never `info_semantic.json` / `mesh_semantic.ply`. Segmentator reads only `mesh.ply`.

**Runtime:** A100 (or T4). Same legacy env as the Mask3D notebook — the [6]/[6b] recipe is identical and battle-tested; Segment3D additionally needs its in-tree Segmentator (`make`) and two source patches (cell [5]).

In [ ]:
# [1] Environment report
import subprocess, platform
print(platform.python_version(), platform.platform())
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

In [ ]:
# [2] Frozen run config (all of this lands in meta.json) — pins per
# docs/c1_m2_protocol.md, filled BEFORE any inference
SCENE = 'room_2'            # STAGE 1. frl/office_0/room_1 ONLY if the gate passes.

S3D_REPO = 'https://github.com/LeapLabTHU/Segment3D'
S3D_COMMIT = 'c510d89a66c372c5358384d6d619f713506214db'   # main @ 2024-12-29
CKPT_GDRIVE_ID = '1Swq9d7rjV2Q1lTuXiKh1z0OZPt9V4sgO'       # official demo ckpt
SEGMENTATOR_REF = '3e5726500896748521a6ceb81271b0f5b2c0e7d2 (vendored in-tree)'

# upstream scripts/run_demo.sh verbatim — do not tune
NUM_QUERIES = 400
TOPK_PER_IMAGE = -1
DBSCAN_EPS = 0.05
DBSCAN_MIN_POINTS = 5
REMOVE_SMALL_GROUP = 15
SEGMENTATOR_KTHRESH = 0.01
SEGMENTATOR_MIN_VERTS = 20

# mask -> dense assignment: frozen contract, PREDECLARED headline threshold
MIN_SCORE = 0.2
MIN_VERTICES = 20

# pinned inputs (tools/replica_scenes.lock.json) — hard gates
MESH_SHA256 = {
    'room_1':          '21695deccc1fe76051d90178eccc1609ee1bab8b5dc715683dd17f7903cf6ee0',
    'room_2':          'e58a7c717c7922e1300ba20ae8053c5dbfdf9bd5f2515e10c71edad98bcb7e44',
    'office_0':        'cdb6ede0b9d455f491ef8fd63cd916a86a505b777842aabd6aa428edf9ff9032',
    'frl_apartment_0': '459374364b1fb6d61b28809fb2ebb722366ffc055caf990a5d659b1ebdd3e71b',
}
N_VERTICES = {'room_1': 645512, 'room_2': 722496, 'office_0': 589517, 'frl_apartment_0': 1757500}
DEVIATIONS = [
    'cuml.cluster.DBSCAN -> sklearn.cluster.DBSCAN (CPU; same eps/min_samples; avoids RAPIDS in the legacy cu113 env)',
    'demo.py pyviz3d visualization replaced by masks/scores export (full-res, original vertex order)',
    'model runs in a dedicated python 3.10 conda env (m3d); Colab kernel untouched',
    'nvcc 11.3 from nvidia/label/cuda-11.3.1 conda channel (subset; libnpp CDN artifact is corrupt)',
    'pip<24.1 + setuptools==69.5.1 (2022 pins have pre-PEP-508 metadata; torch 1.12 needs pkg_resources)',
    'built with gcc-9 (torch CUDA-11.3 gate requires g++ <= 10.0.0; Ubuntu g++-10 is 10.5)',
]

In [ ]:
# [3] Drive -> VM disk (single copies)
from google.colab import drive
drive.mount('/content/drive')
import shutil, pathlib
WORK = pathlib.Path('/content/c1'); (WORK / 'segmenter').mkdir(parents=True, exist_ok=True)
shutil.copy(f'/content/drive/MyDrive/c1/{SCENE}/mesh.ply', WORK / 'mesh.ply')
for f in ('base.py', 'ply.py', 'mask_resolve.py'):
    shutil.copy(f'/content/drive/MyDrive/c1/segmenter/{f}', WORK / 'segmenter' / f)
(WORK / 'segmenter' / '__init__.py').write_text('')
import sys; sys.path.insert(0, str(WORK))

In [ ]:
# [4] Hard gate: pinned mesh hash + vertex count
from segmenter.base import sha256_file
from segmenter.ply import parse_vertices
sha = sha256_file(WORK / 'mesh.ply')
assert sha == MESH_SHA256[SCENE], f'mesh hash mismatch: {sha}'
xyz = parse_vertices(WORK / 'mesh.ply')
assert len(xyz) == N_VERTICES[SCENE], f'{len(xyz)} != {N_VERTICES[SCENE]}'
print(f'{SCENE}: {len(xyz)} vertices, sha256 {sha[:16]}... OK')

In [ ]:
# [5] Clone pinned commit + the two PREDECLARED patches (anchor-asserted)
!git clone {S3D_REPO} /content/segment3d
!cd /content/segment3d && git checkout {S3D_COMMIT}
demo = pathlib.Path('/content/segment3d/demo.py')
src = demo.read_text()

# patch 1: cuml (GPU) -> sklearn (CPU) DBSCAN — same eps/min_samples semantics
a1 = 'from cuml.cluster import DBSCAN'
assert a1 in src, 'patch anchor 1 drifted'
src = src.replace(a1, 'from sklearn.cluster import DBSCAN', 1)
a2 = '''                    DBSCAN(
                        eps=cfg.general.dbscan_eps,
                        min_samples=cfg.general.dbscan_min_points,
                        verbose=2
                    )
                    .fit(raw_coordinates[curr_masks].cuda())
                    .labels_
                )
                clusters = clusters.get()'''
assert a2 in src, 'patch anchor 2 drifted'
src = src.replace(a2, '''                    DBSCAN(
                        eps=cfg.general.dbscan_eps,
                        min_samples=cfg.general.dbscan_min_points,
                    )
                    .fit(np.asarray(raw_coordinates[curr_masks]))
                    .labels_
                )
                clusters = np.asarray(clusters)''', 1)

# patch 2: export masks+scores instead of the pyviz3d visualization
a3 = '    save_visualization(mesh, scores, masks_binary, cfg.general.test_scene, confidence_threshold=0.2)'
assert a3 in src, 'patch anchor 3 drifted'
src = src.replace(a3, '''    out_dir = os.environ.get('S3D_MASK_DIR', '/content/mask_out')
    os.makedirs(out_dir, exist_ok=True)
    np.save(os.path.join(out_dir, 'mesh_masks.npy'), masks_binary.numpy().astype(bool))
    np.save(os.path.join(out_dir, 'mesh_scores.npy'), scores.numpy().astype('float64'))
    print('exported masks', tuple(masks_binary.shape), 'to', out_dir)''', 1)
demo.write_text(src)
print('demo.py patched (sklearn DBSCAN + mask export)')

In [ ]:
# [6] Environment — IDENTICAL to the proven Mask3D recipe (kernel untouched,
# Miniforge, CUDA-11.3 subset without the corrupt libnpp).
%%bash
set -e
rm -rf /opt/conda
wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -O /tmp/mf.sh
bash /tmp/mf.sh -b -p /opt/conda
/opt/conda/bin/conda create -y -q -n m3d -c conda-forge --override-channels python=3.10
/opt/conda/bin/conda install -y -q -n m3d -c "nvidia/label/cuda-11.3.1" --override-channels \
  cuda-nvcc cuda-cudart cuda-thrust cuda-nvrtc libcublas libcusparse libcurand libcusolver cuda-nvtx
/opt/conda/envs/m3d/bin/python --version
/opt/conda/envs/m3d/bin/nvcc --version | tail -1

In [ ]:
# [6b] Model-env installs — proven recipe; pointnet2 from Segment3D's tree,
# Segmentator built with make. NO cuml (patched to sklearn).
PIP = '/opt/conda/envs/m3d/bin/pip'
PY  = '/opt/conda/envs/m3d/bin/python'
E = 'PATH=/opt/conda/envs/m3d/bin:$PATH CC=gcc-9 CXX=g++-9 CUDAHOSTCXX=g++-9 CUDA_HOME=/opt/conda/envs/m3d MAX_JOBS=4'
%cd /content/segment3d
!apt-get -qq install -y libopenblas-dev gcc-9 g++-9 ninja-build > /dev/null
!{PIP} -q install 'pip<24.1'
!{PIP} -q install 'setuptools==69.5.1' wheel ninja==1.10.2.3
!{PIP} -q install torch==1.12.1+cu113 torchvision==0.13.1+cu113 --extra-index-url https://download.pytorch.org/whl/cu113
!{PIP} -q install pytorch-lightning==1.7.2 fire imageio tqdm
!{PIP} -q install python-dotenv pyviz3d scipy plyfile scikit-learn trimesh loguru albumentations volumentations
!{PIP} -q install antlr4-python3-runtime==4.8 omegaconf==2.0.6 hydra-core==1.0.5 --no-deps
!{E} {PIP} -q install --no-build-isolation --no-deps 'git+https://github.com/facebookresearch/detectron2.git@710e7795d0eeadf9def0e7ef957eea13532e34cf'
!{E} {PIP} install -v --no-build-isolation --no-deps -U git+https://github.com/NVIDIA/MinkowskiEngine 2>&1 | tail -5
!{PIP} install --no-build-isolation torch-scatter==2.1.0 -f https://data.pyg.org/whl/torch-1.12.1+cu113.html
!{PIP} -q install open3d==0.16.0 torchmetrics==0.11.0 pycocotools h5py transforms3d fvcore cloudpickle 'Pillow==9.3.0' gorilla-core==0.2.7.8
!cd third_party/pointnet2 && {E} {PIP} install --no-build-isolation .
!cd third_party/Segmentator && make 2>&1 | tail -2 && ls -la segmentator
!{PY} -c "import torch, MinkowskiEngine as ME, pytorch_lightning, detectron2, sklearn; print('torch', torch.__version__, '| ME', ME.__version__, '| pl', pytorch_lightning.__version__, '| cuda_ok', torch.cuda.is_available())"

In [ ]:
# [7] Checkpoint + pin its hash
!mkdir -p /content/segment3d/checkpoints
!pip install -q gdown && gdown {CKPT_GDRIVE_ID} -O /content/segment3d/checkpoints/segment3d.ckpt
from segmenter.base import sha256_file
CKPT_SHA = sha256_file(pathlib.Path('/content/segment3d/checkpoints/segment3d.ckpt'))
print('checkpoint sha256:', CKPT_SHA)

In [ ]:
# [8a] Segmentator preprocessing — reads ONLY mesh.ply geometry (G2 visible
# here): graph-based mesh segmentation, upstream params 0.01 / 20.
# Replica raw meshes are QUAD meshes with 9 vertex properties; the vendored
# tinyply corrupts on that (Segmentator requests triangle lists). Feed a
# geometry-only TRIANGULATED rewrite — x,y,z + tris, SAME vertex order, so
# segIndices align 1:1 with original vertices. Quad diagonals add no
# segment boundaries (near-zero normal difference within a planar quad).
import numpy as np, re, json as _json, os
!rm -rf /content/segment3d/demo_test/{SCENE} && mkdir -p /content/segment3d/demo_test/{SCENE}
shutil.copy(WORK / 'mesh.ply', f'/content/segment3d/demo_test/{SCENE}/mesh.ply')
raw = (WORK / 'mesh.ply').read_bytes()
end = raw.find(b'end_header\n'); body = raw[end+11:]
NV = N_VERTICES[SCENE]
NF = int(re.search(rb'element face (\d+)', raw[:end]).group(1))
VSTRIDE = 27  # x y z nx ny nz float + r g b uchar
k = body[NV*VSTRIDE]
fdt = np.dtype([('n','u1'),('v','<i4',(int(k),))])
faces = np.frombuffer(body, dtype=fdt, count=NF, offset=NV*VSTRIDE)
assert (faces['n'] == k).all(), f'mixed face sizes (first={k})'
v = faces['v']
if k == 3:   tris = v
elif k == 4: tris = np.concatenate([v[:, [0,1,2]], v[:, [0,2,3]]], axis=0)
else:        raise ValueError(f'unsupported face size {k}')
print(f'{NF} faces (k={k}) -> {len(tris)} triangles')
vdt = np.dtype([('x','<f4'),('y','<f4'),('z','<f4'),('nx','<f4'),('ny','<f4'),('nz','<f4'),('r','u1'),('g','u1'),('b','u1')])
verts = np.frombuffer(body, dtype=vdt, count=NV)
hdr = (f'ply\nformat binary_little_endian 1.0\nelement vertex {NV}\n'
       'property float x\nproperty float y\nproperty float z\n'
       f'element face {len(tris)}\nproperty list uchar int vertex_indices\nend_header\n').encode()
xyz = np.zeros(NV, dtype=np.dtype([('x','<f4'),('y','<f4'),('z','<f4')]))
for c in ('x','y','z'): xyz[c] = verts[c]
tri_rec = np.zeros(len(tris), dtype=np.dtype([('n','u1'),('v','<i4',(3,))]))
tri_rec['n'] = 3; tri_rec['v'] = tris
out = pathlib.Path(f'/content/segment3d/demo_test/{SCENE}')
(out / 'mesh_geom.ply').write_bytes(hdr + xyz.tobytes() + tri_rec.tobytes())
!cd /content/segment3d/third_party/Segmentator && ./segmentator ../../demo_test/{SCENE}/mesh_geom.ply {SEGMENTATOR_KTHRESH} {SEGMENTATOR_MIN_VERTS}
segs = _json.load(open(out / 'mesh_geom.0.010000.segs.json'))
assert len(segs['segIndices']) == NV, f"segs {len(segs['segIndices'])} != {NV}"
shutil.copy(out / 'mesh_geom.0.010000.segs.json', out / 'mesh.0.010000.segs.json')
print(f"{len(set(segs['segIndices']))} graph segments over {NV} vertices")
if not any('geometry-only' in d for d in DEVIATIONS):
    DEVIATIONS.append('Segmentator fed geometry-only TRIANGULATED rewrite of mesh.ply (quads split (a,b,c)+(a,c,d); same vertex order) — vendored tinyply corrupts on quad lists + 9-property vertices')

In [ ]:
# [8b] Segment3D inference (upstream run_demo.sh params verbatim; patched
# demo.py exports masks+scores to S3D_MASK_DIR). sklearn DBSCAN over 400
# queries runs on CPU — expect several minutes, not seconds.
import time
MASK_DIR = '/content/mask_out'
!rm -rf {MASK_DIR}
t0 = time.time()
!cd /content/segment3d && OMP_NUM_THREADS=3 S3D_MASK_DIR={MASK_DIR} {PY} demo.py \
  general.experiment_name="c1m2_{SCENE}" \
  general.project_name="demo" \
  general.train_mode=false \
  general.train_on_segments=true \
  model.num_queries={NUM_QUERIES} \
  general.topk_per_image={TOPK_PER_IMAGE} \
  general.use_dbscan=true \
  general.dbscan_eps={DBSCAN_EPS} \
  general.dbscan_min_points={DBSCAN_MIN_POINTS} \
  general.gpus=1 \
  general.checkpoint="checkpoints/segment3d.ckpt" \
  general.test_scene={SCENE} \
  data.remove_small_group={REMOVE_SMALL_GROUP}
RUNTIME_S = time.time() - t0
print(f'inference wall time: {RUNTIME_S:.0f}s'); print(os.listdir(MASK_DIR))

In [ ]:
# [10] Deterministic resolution -> dense assignment (frozen rule,
# PREDECLARED MIN_SCORE=0.2)
import numpy as np
masks = np.load(f'{MASK_DIR}/mesh_masks.npy')     # [K, N] bool
scores = np.load(f'{MASK_DIR}/mesh_scores.npy').astype(float)
assert masks.shape[1] == N_VERTICES[SCENE], f'not full resolution: {masks.shape}'
assert masks.shape[0] == len(scores), f'{masks.shape[0]} masks vs {len(scores)} scores'
from segmenter.mask_resolve import MaskResolveConfig, resolve_masks
cfg = MaskResolveConfig(min_score=MIN_SCORE, min_vertices=MIN_VERTICES)
vertex_instance_ids = resolve_masks(masks, scores, cfg)
n_inst = len(np.unique(vertex_instance_ids[vertex_instance_ids >= 0]))
print(f'{masks.shape[0]} masks -> {n_inst} instances, '
      f'{float((vertex_instance_ids < 0).mean()):.1%} unclaimed')

In [ ]:
# [11]+[12] Export frozen sidecar + RAW masks, tar, single archive to Drive
import json, subprocess
from segmenter.base import SegmentationOutput, save_segmentation_output
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
seg = SegmentationOutput(
    input_mesh_sha256=MESH_SHA256[SCENE], n_vertices=N_VERTICES[SCENE],
    segmenter_name='segment3d_class_agnostic',
    segmenter_version=S3D_COMMIT[:12],
    config_params_json=json.dumps({
        'checkpoint_gdrive_id': CKPT_GDRIVE_ID, 'checkpoint_sha256': CKPT_SHA,
        'segmentator_ref': SEGMENTATOR_REF,
        'segmentator_kthresh': SEGMENTATOR_KTHRESH,
        'segmentator_min_verts': SEGMENTATOR_MIN_VERTS,
        'num_queries': NUM_QUERIES, 'topk_per_image': TOPK_PER_IMAGE,
        'dbscan_eps': DBSCAN_EPS, 'dbscan_min_points': DBSCAN_MIN_POINTS,
        'remove_small_group': REMOVE_SMALL_GROUP,
        **cfg.params(), 'deviations_from_upstream_env': DEVIATIONS,
    }, sort_keys=True),
    vertex_instance_ids=vertex_instance_ids,
    instance_confidence={int(i): float(scores[i])
                         for i in np.unique(vertex_instance_ids) if i >= 0},
    runtime_seconds=RUNTIME_S, hardware=gpu,
).finalize()
bundle = pathlib.Path(f'/content/c1/s3d_bundle_{SCENE}')
save_segmentation_output(seg, bundle)
np.savez_compressed(bundle / 'raw_masks.npz',
                    masks_packed=np.packbits(masks, axis=1),
                    n_vertices=np.int64(masks.shape[1]),
                    scores=scores)
!tar -czf /content/{SCENE}_s3d_bundle.tar.gz -C /content/c1 s3d_bundle_{SCENE}
!mkdir -p /content/drive/MyDrive/c1/out
shutil.copy(f'/content/{SCENE}_s3d_bundle.tar.gz', '/content/drive/MyDrive/c1/out/')
print('saved to Drive:', seg.output_sha256)

## [13] Local side + the gate (not in Colab)

```
tar -xzf room_2_s3d_bundle.tar.gz
python3 tools/c1_run.py <data>/room_2 s3d_bundle_room_2 replica_room_2 --out-dir runs/phase8_c1/m2
python3 tools/c1_failure_classes.py <data>/room_2 s3d_bundle_room_2 --out runs/phase8_c1/m2/room_2_failure_classes.json
```

**Stage-1 gate (predeclared, docs/c1_m2_protocol.md)** vs Mask3D room_2 @0.2: entity R@0.5 ≥ 0.42 · answer recall vs B ≥ 0.44 · precision vs B ≥ 0.90 · support-answer recall > 1/6 · merged+no_proposal ≤ 27 (−20% from 34). **Fail → STOP, no further scenes.**